# 05 Detailanalyse — Top-Standorte mit swissALTI3D (2 m)

**PA1 ZHAW IUNR** | Bächler, Hofstetter, Reichlin | Betreuer: Patrick Laube

Zweistufiger Ansatz:
1. Kantonale SMCA (25 m, DHM25) → Eignungskarte → Top-Standorte identifizieren
2. **Dieses Notebook:** Lokale Detailanalyse (2 m, swissALTI3D) für die vielversprechendsten Standorte

**Voraussetzungen:**
- `02a_preprocessing_constraints.ipynb` → `constraint_mask_s2.tif`
- `03_preprocessing_wlc.ipynb` → normalisierte Eignungsfaktoren
- WLC-Verschneidung → `suitability_s2.tif` (finale Eignungskarte)

**Datenquelle:** swissALTI3D Download-Links (CSV mit 7'539 Kachel-URLs)

## 1. Setup

In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask as rio_mask
from rasterio.transform import from_bounds
from scipy.ndimage import label
import matplotlib.pyplot as plt
import requests
import zipfile
import io
import warnings
warnings.filterwarnings("ignore")

RAW = Path("../data/raw")
PROC = Path("../data/processed")
OUT = Path("../outputs/figures")
ALTI_DIR = RAW / "dem/swissalti3d"
ALTI_DIR.mkdir(parents=True, exist_ok=True)

CRS = "EPSG:2056"
NODATA = -9999.0

# Anzahl Top-Standorte für Detailanalyse
N_TOP = 10
# Puffer um jeden Standort (Meter) für Kachel-Download
SITE_BUFFER = 2000

## 2. Eignungskarte laden & Top-Standorte identifizieren

In [ ]:
# ============================================================
# PLATZHALTER: Eignungskarte aus WLC-Verschneidung laden
# Dieses TIF wird in einem vorherigen Notebook erzeugt
# (AHP-Gewichte × normalisierte Kriterien × Constraint-Maske × Akzeptanz)
# ============================================================

suitability_path = PROC / "suitability_s2.tif"

# --- PLATZHALTER: Falls Eignungskarte noch nicht existiert ---
if not suitability_path.exists():
    print("⚠ suitability_s2.tif noch nicht vorhanden.")
    print("  Dieses Notebook benötigt die finale Eignungskarte aus der WLC-Verschneidung.")
    print("  Verwende vorläufig die Constraint-Maske als Proxy.\n")
    suitability_path = PROC / "constraints/constraint_mask_s2.tif"
# --- ENDE PLATZHALTER ---

with rasterio.open(suitability_path) as src:
    suit = src.read(1)
    suit_transform = src.transform
    suit_profile = src.profile.copy()

print(f"Eignungskarte: {suitability_path.name}")
print(f"Shape: {suit.shape} | Wertebereich: {suit[suit > 0].min():.3f}–{suit[suit > 0].max():.3f}")

In [ ]:
# Zusammenhängende Flächen identifizieren und nach mittlerem Eignungswert ranken
suit_binary = (suit > 0).astype(int)
labeled, n_clusters = label(suit_binary)
print(f"Zusammenhängende Flächen: {n_clusters}")

# Pro Cluster: Fläche, mittlerer Eignungswert, Schwerpunkt
clusters = []
for i in range(1, n_clusters + 1):
    mask_i = labeled == i
    area_ha = mask_i.sum() * 25 * 25 / 1e4  # bei 25m Auflösung
    if area_ha < 10:  # Mindestfläche S2
        continue
    mean_suit = suit[mask_i].mean()
    # Schwerpunkt in Pixelkoordinaten → LV95
    rows, cols = np.where(mask_i)
    cy, cx = rows.mean(), cols.mean()
    ey = suit_transform.f + cy * suit_transform.e
    ex = suit_transform.c + cx * suit_transform.a
    clusters.append({"id": i, "area_ha": area_ha, "mean_suit": mean_suit,
                     "cx_lv95": ex, "cy_lv95": ey, "row": int(cy), "col": int(cx)})

df_clusters = pd.DataFrame(clusters).sort_values("mean_suit", ascending=False).head(N_TOP).reset_index(drop=True)
df_clusters.index += 1
df_clusters.index.name = "Rang"
print(f"\nTop-{N_TOP} Standorte (≥ 10 ha):")
df_clusters[["area_ha", "mean_suit", "cx_lv95", "cy_lv95"]]

## 3. swissALTI3D Kacheln filtern & herunterladen

In [ ]:
# Download-Links laden (CSV mit 7'539 URLs)
alti_csv = RAW / "dem/ch_swisstopo_swissalti3d.csv"
if not alti_csv.exists():
    # Fallback: im Uploads-Ordner suchen
    alti_csv = Path("../ch_swisstopo_swissalti3d-JXNu4KrF.csv")

urls = pd.read_csv(alti_csv, header=None, names=["url"])
# Kachel-Koordinaten aus Dateinamen extrahieren (z.B. 2692-1165 → E=2692km, N=1165km)
urls["kx"] = urls["url"].str.extract(r"_(\d{4})-").astype(int)
urls["ky"] = urls["url"].str.extract(r"-(\d{4})_").astype(int)
# Kacheln decken 1×1 km ab, Koordinaten in km → Bounds in m
urls["minx"] = urls["kx"] * 1000
urls["maxx"] = urls["minx"] + 1000
urls["miny"] = urls["ky"] * 1000
urls["maxy"] = urls["miny"] + 1000

print(f"swissALTI3D Kacheln total: {len(urls)}")
print(f"Koordinatenbereich: E {urls.kx.min()}–{urls.kx.max()} / N {urls.ky.min()}–{urls.ky.max()}")

In [ ]:
# Für jeden Top-Standort die benötigten Kacheln identifizieren
def get_tiles_for_site(cx, cy, buffer_m, tile_df):
    minx = cx - buffer_m
    maxx = cx + buffer_m
    miny = cy - buffer_m
    maxy = cy + buffer_m
    return tile_df[
        (tile_df.maxx > minx) & (tile_df.minx < maxx) &
        (tile_df.maxy > miny) & (tile_df.miny < maxy)
    ]

all_tiles = set()
for _, site in df_clusters.iterrows():
    tiles = get_tiles_for_site(site.cx_lv95, site.cy_lv95, SITE_BUFFER, urls)
    all_tiles.update(tiles.url.tolist())
    print(f"  Standort {_} ({site.cx_lv95:.0f}/{site.cy_lv95:.0f}): {len(tiles)} Kacheln")

print(f"\nTotal benötigte Kacheln: {len(all_tiles)}")
print(f"Geschätzter Download: ~{len(all_tiles) * 15 / 1000:.1f} GB")

In [ ]:
# Download-Funktion
def download_tile(url, out_dir):
    fname = Path(url).stem + ".xyz"
    out_path = out_dir / fname
    if out_path.exists():
        return out_path  # Bereits heruntergeladen
    
    print(f"  Downloading {Path(url).name}...", end=" ")
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    
    # ZIP entpacken
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for member in zf.namelist():
            if member.endswith(".xyz"):
                zf.extract(member, out_dir)
                out_path = out_dir / member
                break
    print(f"OK ({out_path.stat().st_size/1e6:.1f} MB)")
    return out_path

# ============================================================
# PLATZHALTER: Download nur ausführen wenn gewünscht
# Auskommentieren um tatsächlich herunterzuladen
# ============================================================
DOWNLOAD = False  # ← Auf True setzen um Download zu starten

if DOWNLOAD:
    downloaded = []
    for i, url in enumerate(sorted(all_tiles)):
        try:
            path = download_tile(url, ALTI_DIR)
            downloaded.append(path)
        except Exception as e:
            print(f"  ✗ {Path(url).name}: {e}")
        if (i + 1) % 10 == 0:
            print(f"  --- {i+1}/{len(all_tiles)} Kacheln ---")
    print(f"\n{len(downloaded)} Kacheln heruntergeladen")
else:
    print("Download deaktiviert (DOWNLOAD = False)")
    print(f"Setze DOWNLOAD = True um {len(all_tiles)} Kacheln herunterzuladen")
    downloaded = sorted(ALTI_DIR.glob("*.xyz"))
    if downloaded:
        print(f"Bereits vorhanden: {len(downloaded)} Kacheln")

## 4. XYZ → GeoTIFF konvertieren & zusammenfügen

In [ ]:
def xyz_to_raster(xyz_path, resolution=2):
    """XYZ-Punktwolke zu GeoTIFF konvertieren."""
    data = np.loadtxt(xyz_path)
    x, y, z = data[:, 0], data[:, 1], data[:, 2]
    
    minx, maxx = x.min(), x.max()
    miny, maxy = y.min(), y.max()
    width = int(round((maxx - minx) / resolution)) + 1
    height = int(round((maxy - miny) / resolution)) + 1
    
    transform = from_bounds(minx, miny, maxx + resolution, maxy + resolution, width, height)
    raster = np.full((height, width), NODATA, dtype=np.float32)
    
    col = ((x - minx) / resolution).astype(int)
    row = ((maxy - y) / resolution).astype(int)
    valid = (row >= 0) & (row < height) & (col >= 0) & (col < width)
    raster[row[valid], col[valid]] = z[valid]
    
    return raster, transform

# ============================================================
# PLATZHALTER: Nur ausführen wenn Kacheln vorhanden
# ============================================================
xyz_files = sorted(ALTI_DIR.glob("*.xyz"))
if not xyz_files:
    print("⚠ Keine XYZ-Kacheln vorhanden. Download zuerst ausführen (Sektion 3).")
else:
    print(f"Konvertiere {len(xyz_files)} XYZ-Kacheln → GeoTIFF...")
    tif_dir = ALTI_DIR / "tif"
    tif_dir.mkdir(exist_ok=True)
    
    for xyz in xyz_files:
        tif_path = tif_dir / (xyz.stem + ".tif")
        if tif_path.exists():
            continue
        raster, tf = xyz_to_raster(xyz, resolution=2)
        profile = {"driver": "GTiff", "dtype": "float32", "crs": CRS,
                   "transform": tf, "width": raster.shape[1], "height": raster.shape[0],
                   "count": 1, "nodata": NODATA, "compress": "lz4"}
        with rasterio.open(tif_path, "w", **profile) as dst:
            dst.write(raster, 1)
    print(f"  {len(list(tif_dir.glob('*.tif')))} GeoTIFFs erstellt")

## 5. Detailanalyse pro Top-Standort

In [ ]:
def analyze_site(site_row, tile_dir, buffer_m=SITE_BUFFER):
    """Detailanalyse eines Standorts mit 2m-DEM."""
    cx, cy = site_row.cx_lv95, site_row.cy_lv95
    
    # Relevante TIFs laden und mergen
    tif_dir = tile_dir / "tif"
    tifs = sorted(tif_dir.glob("*.tif"))
    if not tifs:
        return None
    
    # Alle TIFs im Puffer-Bereich öffnen
    datasets = []
    for t in tifs:
        src = rasterio.open(t)
        b = src.bounds
        if (b.right > cx - buffer_m and b.left < cx + buffer_m and
            b.top > cy - buffer_m and b.bottom < cy + buffer_m):
            datasets.append(src)
    
    if not datasets:
        return None
    
    # Merge
    mosaic, mosaic_tf = merge(datasets)
    mosaic = mosaic[0]
    for ds in datasets:
        ds.close()
    
    # Clip auf Buffer
    from shapely.geometry import box
    clip_geom = box(cx - buffer_m, cy - buffer_m, cx + buffer_m, cy + buffer_m)
    clip_gdf = gpd.GeoDataFrame(geometry=[clip_geom], crs=CRS)
    
    # DEM-Ableitungen @ 2m
    valid = (mosaic != NODATA) & np.isfinite(mosaic)
    dy, dx = np.gradient(mosaic.astype(float), 2)  # 2m Auflösung
    slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
    aspect = np.degrees(np.arctan2(-dx, dy)) % 360
    
    # Hillshade
    az, alt = np.radians(315), np.radians(45)
    grad = np.sqrt(dx**2 + dy**2)
    hill = (np.cos(alt) * np.cos(np.arctan(grad)) +
            np.sin(alt) * np.sin(np.arctan(grad)) * np.cos(az - np.arctan2(-dx, dy)))
    hill = (hill * 255).clip(0, 255)
    
    return {
        "dem": mosaic, "slope": slope, "aspect": aspect, "hillshade": hill,
        "transform": mosaic_tf, "valid": valid,
        "cx": cx, "cy": cy, "buffer": buffer_m
    }

In [ ]:
# ============================================================
# PLATZHALTER: Analyse nur ausführen wenn Kacheln konvertiert
# ============================================================
tif_dir = ALTI_DIR / "tif"
if not list(tif_dir.glob("*.tif")) if tif_dir.exists() else True:
    print("⚠ Keine konvertierten TIFs vorhanden.")
    print("  Sektionen 3 + 4 zuerst ausführen.")
else:
    results = {}
    for idx, site in df_clusters.iterrows():
        print(f"\nStandort {idx}: E={site.cx_lv95:.0f} N={site.cy_lv95:.0f} ({site.area_ha:.0f} ha)")
        result = analyze_site(site, ALTI_DIR)
        if result:
            v = result["valid"]
            print(f"  DEM 2m: {result['dem'][v].min():.0f}–{result['dem'][v].max():.0f} m")
            print(f"  Slope:  {result['slope'][v].mean():.1f}° mean, {result['slope'][v].max():.1f}° max")
            print(f"  Aspect: {result['aspect'][v].mean():.0f}° mean")
            results[idx] = result
        else:
            print(f"  ⚠ Keine Daten verfügbar")

## 6. Visualisierung — Standort-Steckbriefe

In [ ]:
# ============================================================
# PLATZHALTER: Visualisierung nur wenn Ergebnisse vorhanden
# ============================================================
if not "results" in dir() or not results:
    print("⚠ Keine Analyseergebnisse vorhanden.")
else:
    for idx, result in results.items():
        site = df_clusters.loc[idx]
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Standort {idx} — E={site.cx_lv95:.0f} / N={site.cy_lv95:.0f} | "
                     f"{site.area_ha:.0f} ha | Eignung: {site.mean_suit:.3f}",
                     fontsize=12, fontweight="bold")
        
        v = result["valid"]
        for ax, data, title, cmap in [
            (axes[0], result["hillshade"], "Hillshade (2 m)", "gray"),
            (axes[1], result["dem"], f"Höhe ({result['dem'][v].min():.0f}–{result['dem'][v].max():.0f} m)", "terrain"),
            (axes[2], result["slope"], f"Slope (mean {result['slope'][v].mean():.1f}°)", "YlOrRd"),
            (axes[3], result["aspect"], "Exposition (°)", "hsv"),
        ]:
            masked = np.ma.masked_where(~v, data)
            im = ax.imshow(masked, cmap=cmap, aspect="equal")
            ax.set_title(title, fontsize=9)
            ax.set_axis_off()
            fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
        
        plt.tight_layout()
        fig.savefig(OUT / f"standort_{idx:02d}_detail_2m.png", dpi=150, bbox_inches="tight")
    
    plt.show()
    print(f"\n{len(results)} Standort-Steckbriefe erstellt")

## 7. Vergleich 25 m vs. 2 m

In [ ]:
# ============================================================
# PLATZHALTER: Vergleich DHM25 vs. swissALTI3D
# ============================================================
if not "results" in dir() or not results:
    print("⚠ Keine Analyseergebnisse vorhanden.")
else:
    # DEM @ 25m laden (aus 02a)
    with rasterio.open(PROC / "dem/dem_gr_25m.tif") as src:
        dem_25m = src.read(1)
        tf_25m = src.transform
    with rasterio.open(PROC / "dem/slope_gr_25m.tif") as src:
        slope_25m = src.read(1)
    
    print("Vergleich DHM25 (25 m) vs. swissALTI3D (2 m)\n")
    print(f"{'Standort':<12s} {'Slope 25m':>10s} {'Slope 2m':>10s} {'Diff':>8s} {'Max 2m':>8s}")
    print("-" * 50)
    
    for idx, result in results.items():
        site = df_clusters.loc[idx]
        # 25m Slope am Standort
        row_25 = int((tf_25m.f - site.cy_lv95) / abs(tf_25m.e))
        col_25 = int((site.cx_lv95 - tf_25m.c) / tf_25m.a)
        if 0 <= row_25 < slope_25m.shape[0] and 0 <= col_25 < slope_25m.shape[1]:
            s25 = slope_25m[max(0,row_25-2):row_25+3, max(0,col_25-2):col_25+3]
            s25_mean = s25[s25 != NODATA].mean() if (s25 != NODATA).any() else float("nan")
        else:
            s25_mean = float("nan")
        
        v = result["valid"]
        s2_mean = result["slope"][v].mean()
        s2_max = result["slope"][v].max()
        
        print(f"  {idx:<10d} {s25_mean:>10.1f}° {s2_mean:>10.1f}° {s2_mean-s25_mean:>+8.1f}° {s2_max:>8.1f}°")
    
    print("\n→ Unterschiede zeigen, dass 25m-DEM steile Mikroreliefs glättet.")
    print("  Für kantonale SMCA ausreichend, für Micro-Siting 2m empfohlen.")

## Zusammenfassung

**Zweistufiger Ansatz:**
1. ✓ Kantonale SMCA mit DHM25 (25 m) → Eignungskarte → Top-Standorte
2. ✓ Lokale Detailanalyse mit swissALTI3D (2 m) → Standort-Steckbriefe

**Für den Bericht (Kap. 7 Resultate):**
- Tabelle der Top-10 Standorte mit Fläche, Eignung, Koordinaten
- Standort-Steckbriefe (Hillshade, Höhe, Slope, Aspect @ 2 m)
- Vergleich 25 m vs. 2 m Slope → zeigt Relevanz der Auflösung für Micro-Siting

**Platzhalter in diesem Notebook:**
- [ ] `suitability_s2.tif` aus WLC-Verschneidung (→ verwendet Constraint-Maske als Proxy)
- [ ] `DOWNLOAD = True` setzen um swissALTI3D-Kacheln herunterzuladen
- [ ] Mindestfläche-Filter anpassen (aktuell ≥ 10 ha aus S2)